In [1]:
# %pip install roboflow supervision opencv-python python-dotenv
# %pip install Cython treys phevaluator
# %pip install eval7

In [2]:
# eval7 can be easier to download direct from git and build it from source

This build on
- img_yolo_07_pretrained_monopoly_poker.ipynb

In [3]:
from dotenv import load_dotenv
from roboflow import Roboflow
import supervision as sv
import cv2
import os

In [4]:
load_dotenv()
roboflow_api_key = os.getenv("roboflow_api_key")

In [5]:
rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace().project("poker_vision_3.0-bev73")
model = project.version(1).model

loading Roboflow workspace...
loading Roboflow project...


In [12]:
img_path = '../_data/resource/pokerboard_888poker_v1.jpg'
result = model.predict(img_path, confidence=40, overlap=30).json()
labels = [item["class"] for item in result["predictions"]]

detections = sv.Detections.from_inference(result)

label_annotator = sv.LabelAnnotator()
bounding_box_annotator = sv.BoxAnnotator()

image = cv2.imread(img_path)

annotated_image = bounding_box_annotator.annotate(
    scene=image, detections=detections)
annotated_image = label_annotator.annotate(
    scene=annotated_image, detections=detections, labels=labels)

# sv.plot_image(image=annotated_image, size=(16, 16))

# Parser

In [7]:
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import numpy as np


# ---------------- Dataclasses for typed returns ----------------

@dataclass
class PlayerBet:
    seat_index: int
    bbox: List[float]      # x1,y1,x2,y2
    amount_text: Optional[str] = None  # fill via OCR


@dataclass
class DealerButton:
    seat_index: Optional[int]
    bbox: Optional[List[float]]


@dataclass
class PotInfo:
    bbox: Optional[List[float]]
    amount_text: Optional[str] = None  # fill via OCR
    raked_bbox: Optional[List[float]] = None
    raked_amount_text: Optional[str] = None  # fill via OCR


# ---------------- Main parser ----------------

class PokerTableParser:
    """
    Parse a YOLO 'Detections' object into structured poker table state.
    Expected attributes on 'detections':
        - detections.xyxy : np.ndarray [N,4] (x1,y1,x2,y2)
        - detections.data["class_name"] : np.ndarray [N] of strings
        - (optional) detections.confidence, detections.class_id

    NOTE: This class does *no* OCR. It returns crop boxes so you can OCR amounts outside.
    """

    CARD_CLASSES = {
        "2c","2d","2h","2s","3c","3d","3h","3s","4c","4d","4h","4s",
        "5c","5d","5h","5s","6c","6d","6h","6s","7c","7d","7h","7s",
        "8c","8d","8h","8s","9c","9d","9h","9s",
        "Tc","Td","Th","Ts","Jc","Jd","Jh","Js",
        "Qc","Qd","Qh","Qs","Kc","Kd","Kh","Ks",
        "Ac","Ad","Ah","As"
    }

    HERO_ACTION_CLASSES = {
        "action_timer","call_button","check_button",
        "raise_button","fold_button","all_in_button","bet_input_box"
    }

    TABLE_CLASSES = {
        "player_seat","card_back","dealer_button","stack_size",
        "player_bet","pot_total","raked_pot","vpip"
    }

    def __init__(self, detections, seat_assign_dist_ratio: float = 0.35):
        self.dets = detections
        self.names: np.ndarray = detections.data["class_name"]
        self.xyxy: np.ndarray = detections.xyxy.astype(float)
        self._seat_assign_dist_ratio = seat_assign_dist_ratio

        # Caches for subsets
        self._precomputed: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}

    # -------------------- Public API (your requested functions) --------------------

    def detect_hero_step(self) -> Dict:
        """
        Returns:
            {
              "is_hero_turn": bool,
              "street": "preflop"|"flop"|"turn"|"river"|"unknown"
            }
        """
        hero = self.detect_hero_cards()["hero_seat_index"]
        is_hero_turn = False
        if hero is not None:
            if self._has_any(self.HERO_ACTION_CLASSES):
                # Action UI visible → hero must act
                is_hero_turn = True
            else:
                # fallback heuristic
                bets = self.find_players_bet()
                hero_has_bet = any(b.seat_index == hero for b in bets)
                opp_has_bet = any(b.seat_index != hero for b in bets)
                is_hero_turn = (not hero_has_bet) and opp_has_bet

        street = self.detect_table_cards()["street"]
        return {"is_hero_turn": is_hero_turn, "street": street}

    def detect_hero_cards(self) -> Dict:
        """
        Returns:
            {
              "hero_seat_index": Optional[int],
              "cards": List[str],  # left-to-right (0, 1)
            }
        """
        seat_xyxy, _ = self._subset("player_seat")
        if len(seat_xyxy) == 0:
            return {"hero_seat_index": None, "cards": []}

        hero_idx = int(np.argmax(self._centers(seat_xyxy)[:, 1]))  # bottom-most seat
        face_xy, face_m = self._subset_in(self.CARD_CLASSES)
        assign = self._assign_to_seats(seat_xyxy, face_xy)

        labels = []
        if hero_idx in assign:
            global_face_idx = np.where(face_m)[0]
            hero_face_local = assign[hero_idx]
            labels = [self.names[global_face_idx[i]] for i in hero_face_local]
            # sort by x (left → right)
            if len(hero_face_local) > 1:
                xs = self._centers(face_xy[hero_face_local])[:, 0]
                order = np.argsort(xs)
                labels = [labels[i] for i in order]

        return {"hero_seat_index": hero_idx, "cards": labels}

    def detect_table_cards(self) -> Dict:
        """
        Returns:
            {
              "cards": List[str],  # left-to-right
              "street": "preflop"|"flop"|"turn"|"river"|"unknown"
            }
        """
        face_xy, face_m = self._subset_in(self.CARD_CLASSES)
        seat_xyxy, _ = self._subset("player_seat")

        if len(face_xy) == 0:
            return {"cards": [], "street": "preflop"}

        # exclude cards assigned to seats (hole cards)
        seat_assign = self._assign_to_seats(seat_xyxy, face_xy) if len(seat_xyxy) else {}
        seat_owned = set(i for lst in seat_assign.values() for i in lst)
        table_idx = [i for i in range(len(face_xy)) if i not in seat_owned]

        if len(table_idx) == 0:
            return {"cards": [], "street": "preflop"}

        W, H = self._image_wh(np.vstack([face_xy, seat_xyxy]) if len(seat_xyxy) else face_xy)
        cs = self._centers(face_xy[table_idx])
        # keep a central horizontal band
        band = (cs[:, 1] > H * 0.25) & (cs[:, 1] < H * 0.75)
        table_idx = [i for keep, i in zip(band, table_idx) if keep]

        labels = []
        if table_idx:
            xs = self._centers(face_xy[table_idx])[:, 0]
            global_face_idx = np.where(face_m)[0]
            order = [ti for _, ti in sorted(zip(xs, table_idx))]
            labels = [self.names[global_face_idx[i]] for i in order]

        n = len(labels)
        street = {0: "preflop", 3: "flop", 4: "turn", 5: "river"}.get(n, "unknown")
        return {"cards": labels, "street": street}

    def find_total_pot(self) -> PotInfo:
        """
        Returns crop(s) to OCR:
            PotInfo(bbox=..., amount_text=None, raked_bbox=..., raked_amount_text=None)
        """
        pot_xy, _ = self._subset("pot_total")
        raked_xy, _ = self._subset("raked_pot")
        bbox = self._largest_box(pot_xy)
        rbox = self._largest_box(raked_xy)
        return PotInfo(bbox=bbox, amount_text=None, raked_bbox=rbox, raked_amount_text=None)

    def get_dealer_button_position(self) -> DealerButton:
        """
        Returns:
            DealerButton(seat_index=Optional[int], bbox=Optional[List[float]])
        """
        seat_xyxy, _ = self._subset("player_seat")
        btn_xyxy, _ = self._subset("dealer_button")
        if len(seat_xyxy) == 0 or len(btn_xyxy) == 0:
            return DealerButton(seat_index=None, bbox=None)

        seat_cs = self._centers(seat_xyxy)
        btn_cs = self._centers(btn_xyxy)
        dist = np.linalg.norm(btn_cs[:, None, :] - seat_cs[None, :, :], axis=2)
        bi, si = np.unravel_index(np.argmin(dist), dist.shape)
        return DealerButton(seat_index=int(si), bbox=btn_xyxy[bi].tolist())

    def get_empty_seats(self) -> Dict:
        """
        Returns:
            {"empty_seat_indices": List[int]}
        """
        seat_xyxy, _ = self._subset("player_seat")
        if len(seat_xyxy) == 0:
            return {"empty_seat_indices": []}

        evidence_sets = [
            self._subset("card_back")[0],
            self._subset("stack_size")[0],
            self._subset("player_bet")[0],
            self._subset("vpip")[0],
        ]
        occupied = set()
        for arr in evidence_sets:
            assign = self._assign_to_seats(seat_xyxy, arr)
            for si, lst in assign.items():
                if lst:
                    occupied.add(si)

        empty = [i for i in range(len(seat_xyxy)) if i not in occupied]
        return {"empty_seat_indices": empty}

    def get_so_players(self) -> List[Dict]:
        """
        'Seated players' summary per seat.
        Adds:
          - hole_cards: [] or ['Ah','Kd']
          - hole_known: bool
        Returns a list:
        [
          {
            "seat_index": int,
            "bbox": [x1,y1,x2,y2],
            "is_dealer": bool,
            "has_stack": bool,
            "has_bet_out": bool,
            "has_card_back": bool,
            "vpip": bool,
            "face_up_cards": List[str],   # face-up near seat (usually hero)
            "hole_cards": List[str],      # canonical hole cards (known else [])
            "hole_known": bool,           # True if ranks known
            "occupied": bool
          }, ...
        ]
        """
        seat_xyxy, _ = self._subset("player_seat")
        if len(seat_xyxy) == 0:
            return []

        dealer = self.get_dealer_button_position().seat_index

        card_back_xy, _ = self._subset("card_back")
        stack_xy, _ = self._subset("stack_size")
        bet_xy, _ = self._subset("player_bet")
        vpip_xy, _ = self._subset("vpip")
        face_xy, face_m = self._subset_in(self.CARD_CLASSES)

        m_back = self._assign_to_seats(seat_xyxy, card_back_xy)
        m_stack = self._assign_to_seats(seat_xyxy, stack_xy)
        m_bet = self._assign_to_seats(seat_xyxy, bet_xy)
        m_vpip = self._assign_to_seats(seat_xyxy, vpip_xy)
        m_face = self._assign_to_seats(seat_xyxy, face_xy)

        global_face_idx = np.where(face_m)[0]
        players = []
        for i in range(len(seat_xyxy)):
            face_labels = [self.names[global_face_idx[j]] for j in m_face.get(i, [])]
            has_any = any([
                len(m_back.get(i, [])) > 0,
                len(m_stack.get(i, [])) > 0,
                len(m_bet.get(i, [])) > 0,
                len(face_labels) > 0,
                len(m_vpip.get(i, [])) > 0
            ])
            players.append({
                "seat_index": i,
                "bbox": seat_xyxy[i].tolist(),
                "is_dealer": (dealer == i),
                "has_stack": len(m_stack.get(i, [])) > 0,
                "has_bet_out": len(m_bet.get(i, [])) > 0,
                "has_card_back": len(m_back.get(i, [])) > 0,
                "vpip": len(m_vpip.get(i, [])) > 0,
                "face_up_cards": face_labels,
                "occupied": has_any
            })

        # ---- enrich with hole-cards (hero + unknown backs) ----
        # Default hole info
        for row in players:
            row["hole_cards"] = []
            row["hole_known"] = False

        # hero face-up cards become hero hole cards
        hero_info = self.detect_hero_cards()
        hero_idx = hero_info["hero_seat_index"]
        hero_cards = hero_info["cards"] or []
        if hero_idx is not None and 0 <= hero_idx < len(players) and hero_cards:
            players[hero_idx]["hole_cards"] = hero_cards[:2]
            players[hero_idx]["hole_known"] = True

        # others: if two card backs near seat and no face-up, mark unknown present
        assign_backs = self._assign_to_seats(seat_xyxy, card_back_xy)
        for si, back_idx_list in assign_backs.items():
            if len(back_idx_list) >= 2 and not players[si]["hole_known"]:
                players[si]["hole_cards"] = []      # unknown ranks
                players[si]["hole_known"] = False

        return players

    def find_players_bet(self) -> List[PlayerBet]:
        """
        Returns a list of PlayerBet items (one per visible bet chip/box per seat).
        """
        seat_xyxy, _ = self._subset("player_seat")
        bet_xy, _ = self._subset("player_bet")
        if len(seat_xyxy) == 0 or len(bet_xy) == 0:
            return []

        assign = self._assign_to_seats(seat_xyxy, bet_xy)
        out: List[PlayerBet] = []
        for si, lst in assign.items():
            for ti in lst:
                out.append(PlayerBet(seat_index=si, bbox=bet_xy[ti].tolist(), amount_text=None))
        return out

    # --------------- Convenience snapshot & export for evaluator ---------------

    def to_dict(self) -> Dict:
        return {
            "hero": self.detect_hero_cards(),
            "community": self.detect_table_cards(),
            "pot": self._pot_to_dict(self.find_total_pot()),
            "dealer": self._dealer_to_dict(self.get_dealer_button_position()),
            "players": self.get_so_players(),
            "player_bets": [bet.__dict__ for bet in self.find_players_bet()],
            "hero_step": self.detect_hero_step()
        }

    def export_hand_for_eval(self) -> dict:
        """
        Canonical payload for equity evaluators:
          - hero_cards: ['As','Kd'] or []
          - board: ['Th','9h','2c','Qs','...'] (0..5)
          - opponents: [{'seat_index': i, 'cards': [], 'known': False}, ...]
          - dead: [] (reserved)
        """
        state = self.to_dict()
        players = state.get("players", [])
        hero = state.get("hero", {})
        hero_seat = hero.get("hero_seat_index")

        # hero cards
        hero_cards: List[str] = []
        if hero_seat is not None and 0 <= hero_seat < len(players):
            hero_cards = players[hero_seat].get("hole_cards", []) or hero.get("cards", [])

        # opponents
        opps = []
        for p in players:
            si = p["seat_index"]
            if si == hero_seat:
                continue
            opps.append({
                "seat_index": si,
                "cards": p.get("hole_cards", []),
                "known": p.get("hole_known", False)
            })

        board = state.get("community", {}).get("cards", [])
        return {"hero_cards": hero_cards, "board": board, "opponents": opps, "dead": []}

    # -------------------- Internals & utilities --------------------

    def _subset(self, label: str) -> Tuple[np.ndarray, np.ndarray]:
        key = f"one::{label}"
        if key in self._precomputed:
            return self._precomputed[key]
        m = (self.names == label)
        arr = self.xyxy[m]
        self._precomputed[key] = (arr, m)
        return arr, m

    def _subset_in(self, labels: set) -> Tuple[np.ndarray, np.ndarray]:
        key = f"many::{hash(tuple(sorted(labels)))}"
        if key in self._precomputed:
            return self._precomputed[key]
        m = np.array([n in labels for n in self.names])
        arr = self.xyxy[m]
        self._precomputed[key] = (arr, m)
        return arr, m

    def _has_any(self, labels: set) -> bool:
        _, m = self._subset_in(labels)
        return bool(m.any())

    @staticmethod
    def _centers(xyxy: np.ndarray) -> np.ndarray:
        if len(xyxy) == 0:
            return np.empty((0,2), dtype=float)
        return np.column_stack(((xyxy[:,0]+xyxy[:,2])/2.0, (xyxy[:,1]+xyxy[:,3])/2.0))

    @staticmethod
    def _image_wh(xyxy: np.ndarray) -> Tuple[float, float]:
        W = float(np.max(xyxy[:, [0, 2]])) if len(xyxy) else 1.0
        H = float(np.max(xyxy[:, [1, 3]])) if len(xyxy) else 1.0
        return W, H

    @staticmethod
    def _box_area(b: np.ndarray) -> float:
        return max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

    def _assign_to_seats(self, seat_xyxy: np.ndarray, target_xyxy: np.ndarray) -> Dict[int, List[int]]:
        """
        Assign each target box to the nearest seat if close enough.
        Distance threshold is relative to image diagonal (self._seat_assign_dist_ratio).
        """
        if len(seat_xyxy) == 0 or len(target_xyxy) == 0:
            return {}
        W, H = self._image_wh(np.vstack([seat_xyxy, target_xyxy]))
        diag = (W**2 + H**2) ** 0.5
        seat_cs = self._centers(seat_xyxy)
        targ_cs = self._centers(target_xyxy)

        assign: Dict[int, List[int]] = {i: [] for i in range(len(seat_xyxy))}
        for ti, tc in enumerate(targ_cs):
            dists = np.linalg.norm(seat_cs - tc, axis=1)
            si = int(np.argmin(dists))
            if dists[si] <= self._seat_assign_dist_ratio * diag:
                assign[si].append(ti)
        return assign

    def _largest_box(self, boxes: np.ndarray) -> Optional[List[float]]:
        if len(boxes) == 0:
            return None
        areas = np.array([self._box_area(b) for b in boxes])
        i = int(np.argmax(areas))
        return boxes[i].tolist()

    @staticmethod
    def _pot_to_dict(p: PotInfo) -> Dict:
        return {
            "bbox": p.bbox, "amount_text": p.amount_text,
            "raked_bbox": p.raked_bbox, "raked_amount_text": p.raked_amount_text
        }

    @staticmethod
    def _dealer_to_dict(d: DealerButton) -> Dict:
        return {"seat_index": d.seat_index, "bbox": d.bbox}


# Different evaluaters

In [8]:
# pip install eval7 treys phevaluator

from typing import Dict, List, Optional, Any, Literal
import random
import math
from itertools import combinations

# --- Shared constants ---------------------------------------------------------
RANKS = "23456789TJQKA"
SUITS = "cdhs"  # libraries expect lowercase suits, e.g., 'As', 'Td'


# =============================================================================
# eval7 adapter
# =============================================================================
import eval7

def _cards_to_eval7(cards: List[str]) -> List[eval7.Card]:
    """Convert ['Ah','Td'] -> [Card('Ah'), Card('Td')]"""
    return [eval7.Card(c) for c in cards]

def _fresh_deck_eval7() -> List[eval7.Card]:
    return [eval7.Card(r + s) for r in RANKS for s in SUITS]

def evaluate_with_eval7(
    hand: Dict[str, Any],
    iters: int = 5000,
    rng_seed: Optional[int] = 42
) -> Dict[str, Any]:
    """
    hand: {
      'hero_cards': ['As','Kd'] or [],
      'board':      ['Th','9h','2c','Qs','...'] (0..5),
      'opponents':  [{'seat_index': i, 'cards': [], 'known': False}, ...],
      'dead':       ['7c', 'Jd', ...]
    }
    Returns:
      {
        'hero_equity': float   # [0,1]
        'n_iter': int,
        'deterministic': bool,
        'wins': int,           # strict wins count (MC)
        'ties': int,           # equality with at least one opp (MC)
        'share_sum': float,    # sum of pot shares across iterations (multiway-safe)
        'context': hand,
        'engine': 'eval7'
      }
    """
    if rng_seed is not None:
        random.seed(rng_seed)

    hero = _cards_to_eval7(hand.get("hero_cards", []))
    board_known = _cards_to_eval7(hand.get("board", []))
    dead_cards = _cards_to_eval7(hand.get("dead", []))

    opps = hand.get("opponents", [])
    known_opps = [_cards_to_eval7(o["cards"]) for o in opps if o.get("known") and o.get("cards")]
    n_unknown_opps = sum(1 for o in opps if not o.get("known"))

    # Build deck and remove used/known cards
    deck = _fresh_deck_eval7()
    used = set(hero + board_known + dead_cards)
    for oc in known_opps:
        used.update(oc)
    deck = [c for c in deck if c not in used]

    # Deterministic showdown?
    deterministic = (len(hero) == 2 and len(board_known) == 5 and n_unknown_opps == 0)
    if deterministic:
        hero_score = eval7.evaluate(hero + board_known)
        opp_scores = [eval7.evaluate(oc + board_known) for oc in known_opps]
        # winner(s): highest score wins in eval7
        all_scores = opp_scores + [hero_score]
        best = max(all_scores)
        if hero_score < best:
            share = 0.0
        else:
            winners = sum(1 for s in all_scores if s == best)
            share = 1.0 / winners
        return {
            "hero_equity": share,
            "n_iter": 1,
            "deterministic": True,
            "wins": int(share == 1.0),
            "ties": int(0 < share < 1.0),
            "share_sum": share,
            "context": hand,
            "engine": "eval7",
        }

    # Monte Carlo (unified pattern)
    wins = ties = 0
    share_sum = 0.0
    board_needed = max(0, 5 - len(board_known))

    # Sanity: enough cards left?
    min_needed = n_unknown_opps * 2 + board_needed
    if len(deck) < min_needed:
        # Impossible state; return zero equity but reveal the issue
        return {
            "hero_equity": 0.0,
            "n_iter": 0,
            "deterministic": False,
            "wins": 0,
            "ties": 0,
            "share_sum": 0.0,
            "context": hand,
            "engine": "eval7",
            "warning": f"Not enough cards left in deck (need {min_needed}, have {len(deck)})."
        }

    for _ in range(iters):
        draw = deck[:]  # copy
        random.shuffle(draw)
        di = 0

        # sample unknown opponents
        opp_draws = []
        for o in opps:
            if o.get("known") and o.get("cards"):
                opp_draws.append(_cards_to_eval7(o["cards"]))
            else:
                opp_draws.append([draw[di], draw[di+1]])
                di += 2

        # sample missing board
        b = board_known[:]
        for _k in range(board_needed):
            b.append(draw[di]); di += 1

        hero_score = eval7.evaluate(hero + b)
        opp_scores = [eval7.evaluate(oc + b) for oc in opp_draws]

        # Unified: compare vs best opponent only
        if opp_scores:
            best_opp = max(opp_scores)  # higher is better in eval7
        else:
            best_opp = float("-inf")    # no opponents

        if hero_score > best_opp:
            wins += 1
            share_sum += 1.0
        elif hero_score == best_opp:
            winners = 1 + sum(1 for s in opp_scores if s == best_opp)
            ties += 1
            share_sum += 1.0 / winners
        # else hero loses

    equity = share_sum / iters
    return {
        "hero_equity": equity,
        "n_iter": iters,
        "deterministic": False,
        "wins": wins,
        "ties": ties,
        "share_sum": share_sum,
        "context": hand,
        "engine": "eval7",
    }


# =============================================================================
# Treys adapter
# =============================================================================
# pip install treys
from treys import Card, Evaluator  # lower score = better

def _deck_str() -> List[str]:
    return [r + s for r in RANKS for s in SUITS]

def _str_to_treys(cards: List[str]) -> List[int]:
    return [Card.new(c) for c in cards]

def evaluate_with_treys(
    hand: Dict[str, Any],
    iters: int = 5000,
    rng_seed: Optional[int] = 42,
    *,
    exact: bool = False,
    exact_cap: int = 300_000
) -> Dict[str, Any]:
    """
    Equity via Treys. Monte-Carlo by default; exact enumeration used when `exact=True`
    or when the state space is small enough (< exact_cap combinations).
    Returns: same shape as the other adapters.
    """
    if rng_seed is not None:
        random.seed(rng_seed)

    evaluator = Evaluator()

    hero = _str_to_treys(hand.get("hero_cards", []))
    board_known = _str_to_treys(hand.get("board", []))
    dead = set(_str_to_treys(hand.get("dead", [])))

    opps = hand.get("opponents", [])
    known_opps = [_str_to_treys(o["cards"]) for o in opps if o.get("known") and o.get("cards")]
    n_unknown_opps = sum(1 for o in opps if not o.get("known"))

    # Build deck & remove used
    deck = set(_str_to_treys(_deck_str()))
    used = set(hero) | set(board_known) | dead
    for oc in known_opps:
        used.update(oc)
    deck = list(deck - used)

    # Deterministic showdown?
    deterministic = (len(hero) == 2 and len(board_known) == 5 and n_unknown_opps == 0)
    if deterministic:
        hero_score = evaluator.evaluate(board_known, hero)
        opp_scores = [evaluator.evaluate(board_known, oc) for oc in known_opps]
        # lower is better
        all_scores = opp_scores + [hero_score]
        best = min(all_scores)
        if hero_score > best:
            share = 0.0
        else:
            winners = sum(1 for s in all_scores if s == best)
            share = 1.0 / winners
        return {
            "hero_equity": share, "n_iter": 1, "deterministic": True,
            "wins": int(share == 1.0), "ties": int(0 < share < 1.0),
            "share_sum": share, "context": hand, "engine": "treys"
        }

    # Decide exact vs MC
    board_needed = max(0, 5 - len(board_known))

    # quick feasibility check (consistency with other adapters)
    min_needed = 2 * n_unknown_opps + board_needed
    if len(deck) < min_needed:
        return {
            "hero_equity": 0.0, "n_iter": 0, "deterministic": False,
            "wins": 0, "ties": 0, "share_sum": 0.0, "context": hand,
            "engine": "treys",
            "warning": f"Not enough cards left (need {min_needed}, have {len(deck)})."
        }

    # State-space upper bound using math.comb (avoid building lists)
    combo_upper = 0
    n_deck = len(deck)
    if n_unknown_opps == 0:
        combo_upper = math.comb(n_deck, board_needed)
    elif n_unknown_opps == 1:
        # choose 2 for villain, then choose board_needed from remaining
        combo_upper = math.comb(n_deck, 2) * math.comb(n_deck - 2, board_needed)

    # if tiny state space, do exact enumeration (only safe for <=1 unknown villain, small board fill)
    do_exact = exact or (combo_upper and combo_upper <= exact_cap and n_unknown_opps <= 1 and board_needed <= 3)

    wins = ties = 0
    share_sum = 0.0

    if do_exact and n_unknown_opps == 0:
        # only missing board
        total = math.comb(len(deck), board_needed)
        for bdraw in combinations(deck, board_needed):
            b = board_known + list(bdraw)
            hero_score = evaluator.evaluate(b, hero)
            opp_scores = [evaluator.evaluate(b, oc) for oc in known_opps]
            # Unified: compare vs best opponent only
            if opp_scores:
                best_opp = min(opp_scores)  # lower is better
            else:
                best_opp = float("inf")
            if hero_score < best_opp:
                wins += 1
                share_sum += 1.0
            elif hero_score == best_opp:
                winners = 1 + sum(1 for s in opp_scores if s == best_opp)
                ties += 1
                share_sum += 1.0 / winners

        return {
            "hero_equity": share_sum / total, "n_iter": total, "deterministic": False,
            "wins": wins, "ties": ties, "share_sum": share_sum, "context": hand, "engine": "treys"
        }

    # Otherwise MC
    for _ in range(iters):
        draw = deck[:]
        random.shuffle(draw)
        di = 0

        # sample unknown villains
        opp_draws = []
        for o in opps:
            if o.get("known") and o.get("cards"):
                opp_draws.append(_str_to_treys(o["cards"]))
            else:
                opp_draws.append([draw[di], draw[di+1]]); di += 2

        # sample missing board
        b = board_known[:]
        for _k in range(board_needed):
            b.append(draw[di]); di += 1

        hero_score = evaluator.evaluate(b, hero)
        opp_scores = [evaluator.evaluate(b, oc) for oc in opp_draws]

        # Unified: compare vs best opponent only
        if opp_scores:
            best_opp = min(opp_scores)  # lower is better
        else:
            best_opp = float("inf")

        if hero_score < best_opp:
            wins += 1
            share_sum += 1.0
        elif hero_score == best_opp:
            winners = 1 + sum(1 for s in opp_scores if s == best_opp)
            ties += 1
            share_sum += 1.0 / winners

    return {
        "hero_equity": share_sum / iters, "n_iter": iters, "deterministic": False,
        "wins": wins, "ties": ties, "share_sum": share_sum, "context": hand, "engine": "treys"
    }


# =============================================================================
# PH Evaluator adapter
# =============================================================================
# pip install phevaluator
from phevaluator.evaluator import evaluate_cards  # lower value = stronger

def _deck_str_pheval() -> List[str]:
    return [r + s for r in RANKS for s in SUITS]

def _score_pheval(cards7: List[str]) -> int:
    """cards7 must be exactly 7 strings like 'Ah'."""
    # evaluate_cards takes 7 separate args; splat list
    return evaluate_cards(*cards7)

def evaluate_with_phevaluator(
    hand: Dict[str, Any],
    iters: int = 5000,
    rng_seed: Optional[int] = 42
) -> Dict[str, Any]:
    """
    Equity via PH Evaluator (perfect-hash). Monte-Carlo when unknown info;
    deterministic when everyone’s hole cards and a 5-card board are known.
    Returns same shape as the other adapters.
    """
    if rng_seed is not None:
        random.seed(rng_seed)

    hero = hand.get("hero_cards", [])
    board = hand.get("board", [])
    dead = set(hand.get("dead", []))
    opps = hand.get("opponents", [])
    known_opps = [o["cards"] for o in opps if o.get("known") and o.get("cards")]
    n_unknown_opps = sum(1 for o in opps if not o.get("known"))

    # Build deck & remove used
    deck = set(_deck_str_pheval())
    used = set(hero) | set(board) | dead
    for oc in known_opps:
        used.update(oc)
    deck = list(deck - used)

    # Deterministic showdown?
    deterministic = (len(hero) == 2 and len(board) == 5 and n_unknown_opps == 0)
    if deterministic:
        hero_score = _score_pheval(hero + board)
        opp_scores = [_score_pheval(oc + board) for oc in known_opps]
        # lower is better
        all_scores = opp_scores + [hero_score]
        best = min(all_scores)
        if hero_score > best:
            share = 0.0
        else:
            winners = sum(1 for s in all_scores if s == best)
            share = 1.0 / winners
        return {
            "hero_equity": share, "n_iter": 1, "deterministic": True,
            "wins": int(share == 1.0), "ties": int(0 < share < 1.0),
            "share_sum": share, "context": hand, "engine": "phevaluator"
        }

    # Monte Carlo (unified pattern)
    wins = ties = 0
    share_sum = 0.0
    board_needed = max(0, 5 - len(board))

    # quick feasibility check
    min_needed = 2 * n_unknown_opps + board_needed
    if len(deck) < min_needed:
        return {
            "hero_equity": 0.0, "n_iter": 0, "deterministic": False,
            "wins": 0, "ties": 0, "share_sum": 0.0, "context": hand, "engine": "phevaluator",
            "warning": f"Not enough cards left (need {min_needed}, have {len(deck)})."
        }

    for _ in range(iters):
        draw = deck[:]
        random.shuffle(draw)
        di = 0

        # unknown villains
        opp_draws = []
        for o in opps:
            if o.get("known") and o.get("cards"):
                opp_draws.append(o["cards"])
            else:
                opp_draws.append([draw[di], draw[di+1]]); di += 2

        # missing board
        b = board[:]
        for _k in range(board_needed):
            b.append(draw[di]); di += 1

        hero_score = _score_pheval(hero + b)
        opp_scores = [_score_pheval(oc + b) for oc in opp_draws]

        # Unified: compare vs best opponent only
        if opp_scores:
            best_opp = min(opp_scores)  # lower is better
        else:
            best_opp = float("inf")

        if hero_score < best_opp:
            wins += 1
            share_sum += 1.0
        elif hero_score == best_opp:
            winners = 1 + sum(1 for s in opp_scores if s == best_opp)
            ties += 1
            share_sum += 1.0 / winners

    return {
        "hero_equity": share_sum / iters, "n_iter": iters, "deterministic": False,
        "wins": wins, "ties": ties, "share_sum": share_sum, "context": hand, "engine": "phevaluator"
    }


# =============================================================================
# Unified dispatcher
# =============================================================================
EngineName = Literal["eval7", "treys", "phevaluator"]

def evaluate(
    hand: Dict[str, Any],
    engine: EngineName = "eval7",
    *,
    iters: int = 5000,
    rng_seed: Optional[int] = 42,
    # treys-only knobs (ignored by others)
    exact: bool = False,
    exact_cap: int = 300_000
) -> Dict[str, Any]:
    """
    Unified entrypoint.
    - engine: 'eval7' | 'treys' | 'phevaluator'
    - iters / rng_seed: used by all MC modes
    - exact / exact_cap: forwarded to treys only
    Returns the same dict shape for all engines (plus 'engine' field).
    """
    if engine == "eval7":
        return evaluate_with_eval7(hand, iters=iters, rng_seed=rng_seed)
    elif engine == "treys":
        return evaluate_with_treys(hand, iters=iters, rng_seed=rng_seed,
                                   exact=exact, exact_cap=exact_cap)
    elif engine == "phevaluator":
        return evaluate_with_phevaluator(hand, iters=iters, rng_seed=rng_seed)
    else:
        raise ValueError(f"Unknown engine: {engine}")


In [9]:
sample_hand = {
    "hero_cards": ["As", "Kd"],
    "board": ["Th", "9h", "2c"],
    "opponents": [
        {"seat_index": 1, "known": False, "cards": []},
    ],
    "dead": []
}
print(evaluate(sample_hand, engine="eval7", iters=10000))
print(evaluate(sample_hand, engine="treys", iters=10000))
print(evaluate(sample_hand, engine="phevaluator", iters=10000))

{'hero_equity': 0.50495, 'n_iter': 10000, 'deterministic': False, 'wins': 4992, 'ties': 115, 'share_sum': 5049.5, 'context': {'hero_cards': ['As', 'Kd'], 'board': ['Th', '9h', '2c'], 'opponents': [{'seat_index': 1, 'known': False, 'cards': []}], 'dead': []}, 'engine': 'eval7'}
{'hero_equity': 0.50065, 'n_iter': 10000, 'deterministic': False, 'wins': 4959, 'ties': 95, 'share_sum': 5006.5, 'context': {'hero_cards': ['As', 'Kd'], 'board': ['Th', '9h', '2c'], 'opponents': [{'seat_index': 1, 'known': False, 'cards': []}], 'dead': []}, 'engine': 'treys'}
{'hero_equity': 0.52015, 'n_iter': 10000, 'deterministic': False, 'wins': 5155, 'ties': 93, 'share_sum': 5201.5, 'context': {'hero_cards': ['As', 'Kd'], 'board': ['Th', '9h', '2c'], 'opponents': [{'seat_index': 1, 'known': False, 'cards': []}], 'dead': []}, 'engine': 'phevaluator'}


# PokerHandAnalyzer + DecisionReporter

In [10]:
from __future__ import annotations

import re
import traceback
from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional, Protocol, Sequence, Tuple

# =============================================================================
# Generic helpers (parsing, math, collections)
# =============================================================================

# Strip most currency symbols/whitespace commonly seen in poker UIs.
# We keep digits, minus/plus, dot, and comma for the numeric parse step.
_CCY = re.compile(r"[ \t\n\r,$€£¥₩₽krKRMmA-Za-z]*", re.IGNORECASE)
_NUM = re.compile(r"[-+]?\d*\.?\d+")

def parse_money(text: Optional[str]) -> Optional[float]:
    """
    Parse a noisy money string into a float.
    - Keeps last numeric occurrence (useful if 'Pot: $12.50' etc.)
    - Treats comma as decimal if present.
    Returns None if no number is found or input is falsy.
    """
    if not text:
        return None
    s = _CCY.sub("", text).strip().replace(",", ".")
    m = _NUM.findall(s)
    return float(m[-1]) if m else None

def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def pct_or_qmark(x: Optional[float], places: int = 1) -> str:
    return "?" if x is None else f"{x*100:.{places}f}%"

def median(vals: Sequence[float]) -> float:
    if not vals:
        return 0.0
    s = sorted(vals)
    n = len(s)
    return s[n // 2] if (n % 2) else 0.5 * (s[n // 2 - 1] + s[n // 2])

def last_non_none(items: Iterable[Optional[float]]) -> Optional[float]:
    out: Optional[float] = None
    for v in items:
        if v is not None:
            out = v
    return out

# =============================================================================
# Seating / position utilities
# =============================================================================

def seat_distance(a: int, b: int, n: int) -> int:
    """Clockwise distance (in seat index space) from seat a to seat b modulo n."""
    return (b - a) % n

def position_names(n_players: int) -> List[str]:
    """Return common position names for 6-max / 9-max (BTN-first order)."""
    if n_players <= 0:
        return []
    six_max = ["BTN", "SB", "BB", "UTG", "HJ", "CO"]
    nine_max = ["BTN", "SB", "BB", "UTG", "UTG+1", "MP", "LJ", "HJ", "CO"]
    table = six_max if n_players <= 6 else nine_max
    return table[:n_players]

def order_occupied_by_btn_first(occupied_seat_indices: List[int],
                                dealer_seat: Optional[int]) -> List[int]:
    """
    Return occupied seat indices ordered clockwise with BTN (dealer) first if known.
    Assumes seat_index increases clockwise (typical numeric seat IDs).
    """
    if not occupied_seat_indices:
        return []
    ordered = sorted(occupied_seat_indices)
    if dealer_seat is None or dealer_seat not in ordered:
        return ordered
    start = ordered.index(dealer_seat)
    return ordered[start:] + ordered[:start]

def first_to_act_seat(order_btn_first: List[int], street: str) -> Optional[int]:
    """
    Determine the seat index of the first player to act given BTN-first order.
    - Preflop: player to the left of the BB (i.e., first after BB).
    - Postflop: player to the left of BTN (i.e., SB if present).
    Returns None if not enough players to determine.
    """
    n = len(order_btn_first)
    if n < 2:
        return None
    if street == "preflop":
        if n == 2:
            # Heads-up preflop: BTN posts SB; first to act is BTN (order[0]).
            return order_btn_first[0]
        if n < 3:
            return None
        bb = order_btn_first[2]
        idx_bb = order_btn_first.index(bb)
        return order_btn_first[(idx_bb + 1) % n]
    # Postflop: first seat to the left of BTN
    return order_btn_first[1]

def seat_to_position_map(order_btn_first: List[int]) -> Dict[int, str]:
    """Map seat index -> position name based on BTN-first seat order."""
    names = position_names(len(order_btn_first))
    return {si: names[i] for i, si in enumerate(order_btn_first)}

# =============================================================================
# Table state extraction & numeric transforms (pure functions)
# =============================================================================

def occupied_seat_indices_from_players(players: List[Dict[str, Any]]) -> List[int]:
    return [int(p["seat_index"]) for p in players if p.get("occupied") and "seat_index" in p]

def aggregate_bets_by_seat(player_bets: List[Dict[str, Any]]) -> Dict[int, float]:
    acc: Dict[int, float] = {}
    for b in player_bets:
        si = b.get("seat_index")
        if si is None:
            continue
        amt = parse_money(b.get("amount_text")) or 0.0
        acc[si] = acc.get(si, 0.0) + amt
    return acc

def stacks_by_seat(players: List[Dict[str, Any]]) -> Dict[int, Optional[float]]:
    out: Dict[int, Optional[float]] = {}
    for p in players:
        si = p.get("seat_index")
        if si is None:
            continue
        stack_txt = p.get("stack_text")
        out[si] = parse_money(stack_txt) if stack_txt is not None else None
    return out

def compute_to_call_from_bets(hero_seat: Optional[int],
                              occupied_indices: Sequence[int],
                              bets_by_seat: Dict[int, float]) -> Optional[float]:
    """Approximate to-call as max(other_bets) - hero_bet, floored at 0."""
    if hero_seat is None:
        return None
    hero_amt = bets_by_seat.get(hero_seat, 0.0)
    other_bets = [bets_by_seat.get(si, 0.0) for si in occupied_indices if si != hero_seat]
    if not other_bets:
        return None
    diff = max(other_bets) - hero_amt
    return max(0.0, diff)

def compute_effective_stack_from_stacks(hero_seat: Optional[int],
                                        stacks: Dict[int, Optional[float]]) -> Optional[float]:
    """Min(hero_stack, max(opponent_stack)) if both known; otherwise None."""
    if hero_seat is None:
        return None
    hero_stack = stacks.get(hero_seat)
    if hero_stack is None:
        return None
    opp_stacks = [s for si, s in stacks.items() if si != hero_seat and s is not None]
    return min(hero_stack, max(opp_stacks)) if opp_stacks else None

def compute_pot_odds(pot: Optional[float], to_call: Optional[float]) -> Optional[float]:
    if pot is None or to_call is None:
        return None
    denom = pot + to_call
    return (to_call / denom) if denom > 0 else None

def compute_spr(effective_stack: Optional[float], pot: Optional[float]) -> Optional[float]:
    if effective_stack is None or pot is None or pot <= 0:
        return None
    return effective_stack / pot

def normalize_street(s: Any) -> str:
    return str(s or "unknown").strip().lower()

def is_facing_bet(hero_seat: Optional[int],
                  occupied_indices: Sequence[int],
                  bets_by_seat: Dict[int, float]) -> bool:
    return any((si != hero_seat) and (bets_by_seat.get(si, 0.0) > 0.0) for si in occupied_indices)

def hero_has_bet(hero_seat: Optional[int], bets_by_seat: Dict[int, float]) -> bool:
    return (hero_seat is not None) and (bets_by_seat.get(hero_seat, 0.0) > 0.0)

# =============================================================================
# Core Data
# =============================================================================

@dataclass
class FeaturePack:
    n_seats: int
    n_players: int
    street: str
    hero_seat: Optional[int]
    hero_pos: str
    dealer_seat: Optional[int]
    community: List[str]
    pot: Optional[float]
    raked_pot: Optional[float]
    to_call: Optional[float]
    pot_odds: Optional[float]
    effective_stack: Optional[float]
    spr: Optional[float]
    facing_bet: bool
    multiway: bool
    first_to_act: bool

# =============================================================================
# Analyzer
# =============================================================================

class PokerHandAnalyzer:
    """
    Consumes a state dict from PokerTableParser.to_dict() and derives:
    - Normalized, numeric features for decision logic (FeaturePack).
    - Human-readable summary (text) and compact JSON for downstream systems.
    Expected 'state' keys (best effort; missing values allowed).
    """

    def __init__(self, state: Dict[str, Any]):
        self.s = state
        self.players: List[Dict[str, Any]] = state.get("players", [])
        self.hero: Dict[str, Any] = state.get("hero", {})
        self.community: Dict[str, Any] = state.get("community", {})
        self.dealer: Dict[str, Any] = state.get("dealer", {})
        self.bets: List[Dict[str, Any]] = state.get("player_bets", [])
        self.pot_info: Dict[str, Any] = state.get("pot", {})

        # Parse monetary fields (optional)
        self._pot_total: Optional[float] = parse_money(self.pot_info.get("amount_text"))
        self._raked_total: Optional[float] = parse_money(self.pot_info.get("raked_amount_text"))

        # Quick references
        self._seat_count: int = len(self.players)
        self._occupied_seats: List[Dict[str, Any]] = [p for p in self.players if p.get("occupied")]
        self._hero_idx: Optional[int] = self.hero.get("hero_seat_index")
        self._dealer_idx: Optional[int] = self.dealer.get("seat_index")

        # Aggregates
        self._bets_by_seat: Dict[int, float] = aggregate_bets_by_seat(self.bets)
        self._stacks_by_seat: Dict[int, Optional[float]] = stacks_by_seat(self.players)

    # ------------------------------ computation ------------------------------

    def _seat_order_and_positions(self) -> Tuple[List[int], Dict[int, str]]:
        occ_indices = occupied_seat_indices_from_players(self.players)
        order_btn_first = order_occupied_by_btn_first(occ_indices, self._dealer_idx)
        seat_to_pos = seat_to_position_map(order_btn_first)
        return order_btn_first, seat_to_pos

    def compute_features(self) -> FeaturePack:
        street = normalize_street(self.community.get("street", "unknown"))
        hero_seat = self._hero_idx
        dealer_seat = self._dealer_idx

        # Positions
        order_btn_first, seat_to_pos = self._seat_order_and_positions()
        hero_pos = seat_to_pos.get(hero_seat, "NA")

        # Participants
        n_seats = self._seat_count
        n_players = len(order_btn_first)
        multiway = n_players >= 3

        # Action context
        occ_indices = order_btn_first or occupied_seat_indices_from_players(self.players)
        facing = is_facing_bet(hero_seat, occ_indices, self._bets_by_seat)
        hero_bet = hero_has_bet(hero_seat, self._bets_by_seat)

        # First to act
        fta_seat = first_to_act_seat(order_btn_first, street)
        first_to_act_flag = (fta_seat == hero_seat) if (hero_seat is not None and fta_seat is not None) else False

        # Amounts
        pot = self._pot_total
        to_call = None
        if facing and not hero_bet:
            to_call = compute_to_call_from_bets(hero_seat, occ_indices, self._bets_by_seat)
        pot_odds = compute_pot_odds(pot, to_call)

        # Stacks / SPR
        effective_stack = compute_effective_stack_from_stacks(hero_seat, self._stacks_by_seat)
        spr = compute_spr(effective_stack, pot)

        return FeaturePack(
            n_seats=n_seats,
            n_players=n_players,
            street=street,
            hero_seat=hero_seat,
            hero_pos=hero_pos,
            dealer_seat=dealer_seat,
            community=list(self.community.get("cards", [])),
            pot=pot,
            raked_pot=self._raked_total,
            to_call=to_call,
            pot_odds=pot_odds,
            effective_stack=effective_stack,
            spr=spr,
            facing_bet=facing,
            multiway=multiway,
            first_to_act=first_to_act_flag,
        )

    # ------------------------------ outputs ------------------------------

    def summary_json(self) -> Dict[str, Any]:
        f = self.compute_features()
        hero_cards = self.s.get("hero", {}).get("cards", [])
        return {
            "street": f.street,
            "community": f.community,
            "hero": {"seat": f.hero_seat, "position": f.hero_pos, "cards": hero_cards},
            "dealer_seat": f.dealer_seat,
            "players": f.n_players,
            "pot": f.pot,
            "to_call": f.to_call,
            "pot_odds": f.pot_odds,
            "spr": f.spr,
            "facing_bet": f.facing_bet,
            "multiway": f.multiway,
            "first_to_act": f.first_to_act,
        }

    def summary_text(self) -> str:
        f = self.compute_features()
        cards = " ".join(self.s.get("hero", {}).get("cards", [])) or "?? ??"
        board = " ".join(f.community) or "(no board)"
        pot = f"{f.pot:.2f}" if f.pot is not None else "?"
        to_call = f"{f.to_call:.2f}" if f.to_call is not None else "0.00"
        odds = f"{100 * f.pot_odds:.1f}%" if f.pot_odds is not None else "?"
        spr = f"{f.spr:.2f}" if f.spr is not None else "?"
        facing = "facing a bet" if f.facing_bet else "no bet faced"
        players_txt = "multiway" if f.multiway else "heads-up"
        return (
            f"[{f.street.upper()}] Hero {f.hero_pos} (seat {f.hero_seat}) — "
            f"hole cards: {cards} | Board: {board} | Pot: {pot} | "
            f"To call: {to_call} (pot odds {odds}) | SPR {spr} | "
            f"{players_txt} | {facing}"
        )

    # ------------------------------ toy policy ------------------------------

    def suggest_next_move(self) -> Dict[str, Any]:
        """
        Very basic heuristic stub (replace with your strategy engine):
        - If not facing a bet:
            • On flop/turn with low SPR (<= 3): small c-bet (33% pot).
            • Otherwise: check.
        - If facing a bet:
            • Call if pot odds <= 0.25 (tighten to 0.20 when multiway), else fold.
        """
        f = self.compute_features()
        action: str = "check"
        size: Optional[str] = None
        reasons: List[str] = []

        if not f.facing_bet:
            if f.street in ("flop", "turn") and (f.spr is not None and f.spr <= 3):
                action, size = "bet", "33% pot"
                reasons.append("low SPR favors aggression")
            else:
                reasons.append("no bet faced; neutral default")
        else:
            thresh = 0.20 if f.multiway else 0.25
            if f.pot_odds is not None and f.pot_odds <= thresh:
                action = "call"
                reasons.append(f"pot odds {f.pot_odds:.2f} <= threshold {thresh:.2f}")
            else:
                action = "fold"
                reasons.append("insufficient pot odds")

        return {
            "action": action,
            "size": size,
            "why": "; ".join(reasons),
            "context": self.summary_json(),
        }

# =============================================================================
# Reporter Interfaces and Results
# =============================================================================

class FeatureLike(Protocol):
    street: str
    pot: Optional[float]
    to_call: Optional[float]
    pot_odds: Optional[float]
    spr: Optional[float]
    multiway: bool
    facing_bet: bool
    hero_pos: str

@dataclass
class EngineResult:
    hero_equity: float
    n_iter: int
    engine: str
    deterministic: bool
    wins: int
    ties: int
    share_sum: float
    warning: Optional[str] = None
    error: Optional[str] = None
    raw: Optional[Dict[str, Any]] = None

@dataclass
class DecisionReport:
    engines: Dict[str, EngineResult]
    consensus_equity: float            # [0..1]
    dispersion: float                  # max(eq) - min(eq)
    features: Dict[str, Any]           # PokerHandAnalyzer.summary_json()
    recommendation: Dict[str, Any]     # {action, size, confidence, why: [...]}
    text: str                          # pretty report

# =============================================================================
# Reporter utilities
# =============================================================================

def normalize_engine_payload(name: str, payload: Dict[str, Any]) -> EngineResult:
    eq = float(payload.get("hero_equity", 0.0))
    return EngineResult(
        hero_equity=clamp(eq, 0.0, 1.0),
        n_iter=int(payload.get("n_iter", 0)),
        engine=str(payload.get("engine", name)),
        deterministic=bool(payload.get("deterministic", False)),
        wins=int(payload.get("wins", 0)),
        ties=int(payload.get("ties", 0)),
        share_sum=float(payload.get("share_sum", 0.0)),
        warning=payload.get("warning"),
        raw=payload
    )

def safe_run_engine(
    label: str,
    fn: Callable[..., Dict[str, Any]],
    hand: Dict[str, Any],
    *,
    iters: int,
    rng_seed: int
) -> EngineResult:
    try:
        out = fn(hand, iters=iters, rng_seed=rng_seed)
        return normalize_engine_payload(label, out)
    except Exception as e:
        return EngineResult(
            hero_equity=0.0, n_iter=0, engine=label, deterministic=False,
            wins=0, ties=0, share_sum=0.0, warning=None,
            error=f"{type(e).__name__}: {e}", raw={"traceback": traceback.format_exc()}
        )

# =============================================================================
# DecisionReporter
# =============================================================================

class DecisionReporter:
    """
    Builds a decision report by running up to three engines, fusing a consensus,
    comparing to pot odds / context, and returning a recommendation.
    Works with evaluate_with_eval7 / evaluate_with_treys / evaluate_with_phevaluator.
    """
    def __init__(
        self,
        mc_iters: int = 20_000,
        rng_seed: int = 7,
        *,
        eval7_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        treys_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        pheval_fn: Optional[Callable[..., Dict[str, Any]]] = None,
    ) -> None:
        self.mc_iters = mc_iters
        self.rng_seed = rng_seed

        g = globals()
        self._eval7_fn = eval7_fn or g.get("evaluate_with_eval7")
        self._treys_fn = treys_fn or g.get("evaluate_with_treys")
        self._pheval_fn = pheval_fn or g.get("evaluate_with_phevaluator")

    def build(self, hand_for_eval: Dict[str, Any], analyzer: Any) -> DecisionReport:
        """
        hand_for_eval: e.g., parser.export_hand_for_eval()
        analyzer: PokerHandAnalyzer(state) instance built on the same frame.
        """
        # Defensive copy + required defaults
        hand = dict(hand_for_eval)
        hand.setdefault("dead", [])

        # Features / JSON
        feat: FeatureLike = analyzer.compute_features()
        features_json: Dict[str, Any] = analyzer.summary_json()

        # Run engines (only those present)
        engines: Dict[str, EngineResult] = {}
        if self._eval7_fn:
            engines["eval7"] = safe_run_engine("eval7", self._eval7_fn, hand, iters=self.mc_iters, rng_seed=self.rng_seed)
        if self._treys_fn:
            engines["treys"] = safe_run_engine("treys", self._treys_fn, hand, iters=self.mc_iters, rng_seed=self.rng_seed)
        if self._pheval_fn:
            engines["phevaluator"] = safe_run_engine("phevaluator", self._pheval_fn, hand, iters=self.mc_iters, rng_seed=self.rng_seed)

        if not engines:
            raise ValueError("No equity engines found. Ensure adapter functions are defined or passed into DecisionReporter.")

        # Consensus & dispersion
        eqs = [e.hero_equity for e in engines.values() if e.error is None]
        consensus = median(eqs)
        dispersion = (max(eqs) - min(eqs)) if eqs else 0.0

        # Recommendation + pretty text
        reco = self._recommend(consensus, dispersion, feat)
        text = self._pretty(consensus, dispersion, feat, engines, reco)

        return DecisionReport(
            engines=engines,
            consensus_equity=consensus,
            dispersion=dispersion,
            features=features_json,
            recommendation=reco,
            text=text
        )

    # ---------------- Internals: policy ----------------
    def _recommend(self, equity: float, dispersion: float, f: FeatureLike) -> Dict[str, Any]:
        """
        Interpretable baseline:
        - Facing a bet → compare equity to pot odds with ±3pp buffer. Raise only when SPR ≤3 and not multiway.
        - No bet → on flop/turn, with SPR ≤3 and equity ≥52% → small bet, else check.
        """
        reasons: List[str] = []
        action: str = "check"
        size: Optional[str] = None

        street = normalize_street(f.street)
        pot, to_call, pot_odds, spr = f.pot, f.to_call, f.pot_odds, f.spr
        multiway, hero_pos = f.multiway, getattr(f, "hero_pos", "NA")

        # Confidence starts high; penalize by disagreement & missing inputs
        confidence = 0.95
        confidence -= clamp(dispersion * 2.0, 0.0, 0.6)
        if pot is None: confidence -= 0.10
        if f.facing_bet and to_call is None: confidence -= 0.20
        if f.facing_bet and pot_odds is None: confidence -= 0.10
        if multiway: confidence -= 0.05
        confidence = clamp(confidence, 0.10, 0.95)

        margin = 0.03  # ±3 percentage points
        eq_txt = pct_or_qmark(equity)
        po_txt = pct_or_qmark(pot_odds)

        if f.facing_bet:
            if (pot is None) or (to_call is None):
                action = "call" if street in ("flop", "turn") else "fold"
                reasons += [
                    "missing pot/to-call; defaulting conservatively",
                    f"equity={eq_txt}, street={street}, pos={hero_pos}",
                ]
            else:
                required = pot_odds or 0.0
                diff = equity - required
                if diff < -margin:
                    action = "fold"
                    reasons.append(f"equity {eq_txt} < required {po_txt} by {abs(diff)*100:.1f}pp (pot odds)")
                elif diff <= margin:
                    action = "call"
                    reasons.append(f"equity {eq_txt} ≈ required {po_txt} (±{margin*100:.0f}pp)")
                else:
                    if (spr is not None and spr <= 3.0) and (not multiway):
                        action, size = "raise", "70% pot"
                        reasons.append(f"edge over pot odds ({eq_txt} > {po_txt}) with low SPR {spr:.2f}")
                        reasons.append("heads-up → value raise")
                    else:
                        action = "call"
                        reasons.append(f"edge over pot odds ({eq_txt} > {po_txt}) but SPR/multiway suggest caution")
        else:
            if street in ("flop", "turn"):
                if (spr is not None and spr <= 3.0) and (equity >= 0.52):
                    action, size = "bet", "33% pot"
                    reasons.append(f"no bet faced; {eq_txt} with low SPR {spr:.2f} → apply pressure")
                elif (equity >= 0.58) and (not multiway):
                    action, size = "bet", "33% pot"
                    reasons.append(f"heads-up with strong equity ({eq_txt}) → value/protection")
                else:
                    action = "check"
                    reasons.append("no bet faced; realize equity")
            elif street == "preflop":
                action = "check"
                reasons.append("preflop without open/raise context → conservative default")
            else:
                # river default
                if equity >= 0.65 and not multiway:
                    action, size = "bet", "33% pot"
                    reasons.append(f"river with high equity ({eq_txt}) → thin value")
                else:
                    action = "check"
                    reasons.append("river with marginal edge → avoid thin bluffs")

        if dispersion >= 0.10:
            reasons.append(f"engine dispersion high ({dispersion*100:.1f}pp)")

        if pot is None and action in ("bet", "raise"):
            size = None
            reasons.append("unknown pot size → omit numeric sizing")

        return {
            "action": action,
            "size": size,
            "confidence": round(confidence, 2),
            "why": reasons,
        }

    # ---------------- Internals: pretty output ----------------
    def _pretty(
        self,
        equity: float,
        dispersion: float,
        f: FeatureLike,
        engines: Dict[str, EngineResult],
        reco: Dict[str, Any],
    ) -> str:
        lines: List[str] = []
        spr_txt = f"{f.spr:.2f}" if f.spr is not None else "?"
        lines.append(
            f"[{f.street.upper()}] "
            f"{'multiway' if f.multiway else 'heads-up'}, "
            f"pos={getattr(f,'hero_pos','NA')}, SPR={spr_txt}"
        )

        def _eline(label: str) -> str:
            e = engines.get(label)
            if not e:
                return f"{label}: (not run)"
            if e.error:
                return f"{label}: ERROR ({e.error.split(':', 1)[0]})"
            return f"{label}: {pct_or_qmark(e.hero_equity)}"

        lines.append("Equity — " + " | ".join([
            _eline("eval7"), _eline("treys"), _eline("phevaluator")
        ]))

        lines.append(f"Consensus: {pct_or_qmark(equity)}  |  Dispersion: {dispersion*100:.1f}pp")

        if f.pot is not None and f.to_call is not None:
            lines.append(f"Pot: {f.pot:.2f}   To call: {f.to_call:.2f}   Pot odds: {pct_or_qmark(f.pot_odds)}")
        else:
            lines.append("Pot / to-call: incomplete (OCR missing)")

        action = (reco.get("action") or "?").upper()
        size = reco.get("size")
        conf = reco.get("confidence", 0.0)
        lines.append(f"→ Recommendation: **{action}**" + (f" ({size})" if size else "") + f"  | confidence {int(conf*100)}%")

        for r in reco.get("why", []):
            lines.append(f"- {r}")

        warns = [f"{k}: {e.warning}" for k, e in engines.items() if e.warning]
        errs = [f"{k}: {e.error}" for k, e in engines.items() if e.error]
        if warns:
            lines.append("Warnings:")
            lines += [f"- {w}" for w in warns]
        if errs:
            lines.append("Errors:")
            lines += [f"- {er}" for er in errs]

        return "\n".join(lines)


In [11]:
# 1) Parse detections
parser = PokerTableParser(detections)
state = parser.to_dict()

# (Optional) Fill OCR numbers here:
# state["pot"]["amount_text"] = "$12.50"
# for b in state["player_bets"]: b["amount_text"] = "$2.50"
# for p in state["players"]: p["stack_text"] = "$75.00"

# 2) Build analyzer + hand for evaluators
analyzer = PokerHandAnalyzer(state)
hand = parser.export_hand_for_eval()
# hand["dead"] = ["Jd"]  # optional, if you saw an exposed/mucked card

# 3) Make the decision report
reporter = DecisionReporter(mc_iters=20000, rng_seed=7)
report = reporter.build(hand, analyzer)

print(report.text)       # pretty human-readable
# Or consume machine-friendly fields:
# report.consensus_equity, report.dispersion, report.recommendation, report.features, report.engines


[PREFLOP] multiway, pos=BB, SPR=?
Equity — eval7: 14.9% | treys: 14.2% | phevaluator: 15.0%
Consensus: 14.9%  |  Dispersion: 0.8pp
Pot / to-call: incomplete (OCR missing)
→ Recommendation: **CHECK**  | confidence 78%
- preflop without open/raise context → conservative default
